# PBIT Split & Augmentation

Prepares the training and test sets for model development.

**Steps**
1. Grouped train/test split by `Serial_Number` (stratified by `Fault_Mode`)
2. Split diagnostics: class counts, severity counts, F06 coverage
3. Gaussian noise augmentation for F04, F05, F06 — training set only
4. Save outputs

**Outputs**
| File | Contents |
|------|----------|
| `data/train_real.csv` | Real training features (no augmentation) |
| `data/test_real.csv` | Real test features (never touched) |
| `data/train_augmented.csv` | Training features: real + synthetic |
| `data/augmented_only.csv` | Synthetic rows only (for inspection) |

## Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pbit.data_validation import load_long_dataset
from pbit.features import extract_features
from pbit.splitting import grouped_train_test_split, split_diagnostics, f06_split_report
from pbit.augmentation import augment_training_set, augmentation_report

FEATURES_PATH  = PROJECT_ROOT / "data" / "PBIT_features_r1.csv"
LONG_CLEAN_PATH = PROJECT_ROOT / "data" / "PBIT_atuador_DESCARAC_r1_long_clean.csv"
PROJECT_ROOT

## Load Data

In [ ]:
import pandas as pd

features_df = pd.read_csv(FEATURES_PATH)
long_df     = load_long_dataset(LONG_CLEAN_PATH)

print(f"Feature matrix : {features_df.shape[0]:,} experiments × {features_df.shape[1]} columns")
print(f"Long dataset   : {long_df.shape[0]:,} rows, {long_df['Experiment_Index'].nunique():,} experiments")

## Train / Test Split

Grouped by `Serial_Number`, stratified by `Fault_Mode`.  
With `n_splits=5` the test set is ~20 % of experiments.

In [ ]:
train_df, test_df = grouped_train_test_split(features_df, n_splits=5, random_state=42)

print(f"Train : {len(train_df):,} experiments  ({len(train_df)/len(features_df)*100:.1f} %)")
print(f"Test  : {len(test_df):,} experiments  ({len(test_df)/len(features_df)*100:.1f} %)")
print(f"Train serial numbers : {train_df['Serial_Number'].nunique()}")
print(f"Test  serial numbers : {test_df['Serial_Number'].nunique()}")

### Class distribution

In [ ]:
split_diagnostics(train_df, test_df)

### Severity distribution

In [ ]:
sev_train = train_df["Severity_Index"].value_counts().sort_index().rename("Train")
sev_test  = test_df["Severity_Index"].value_counts().sort_index().rename("Test")
sev = pd.concat([sev_train, sev_test], axis=1)
sev["Total"]   = sev["Train"] + sev["Test"]
sev["Train_%"] = (sev["Train"] / sev["Total"] * 100).round(1)
sev["Test_%"]  = (sev["Test"]  / sev["Total"] * 100).round(1)
sev

### F06 coverage

F06 has only 7 real experiments across 4 serial numbers.  
At least 1 serial number must appear in each partition.

In [ ]:
f06_split_report(train_df, test_df)

### Serial numbers per partition

In [ ]:
train_sns = sorted(train_df["Serial_Number"].unique())
test_sns  = sorted(test_df["Serial_Number"].unique())
overlap   = set(train_sns) & set(test_sns)

print(f"Train SNs ({len(train_sns)}): {train_sns}")
print()
print(f"Test  SNs ({len(test_sns)}): {test_sns}")
print()
print(f"Overlap (must be empty): {overlap}")

## Augmentation

Generate synthetic experiments for F04, F05, F06 using 1 % Gaussian noise.  
Source signals come only from the **training set**. Test set is untouched.  
Target: 160 training experiments per minority class.

In [ ]:
import time

train_ids = set(train_df["Experiment_Index"].tolist())

t0 = time.time()
aug_long = augment_training_set(
    long_df,
    train_experiment_ids=train_ids,
    target_per_class=160,
    noise_scale=0.01,
    random_state=42,
)
print(f"Augmented long rows generated: {len(aug_long):,}  ({time.time()-t0:.1f}s)")
print(f"Synthetic experiments        : {aug_long['Experiment_Index'].nunique():,}")

In [ ]:
t0 = time.time()
aug_features = extract_features(aug_long)
print(f"Feature extraction: {time.time()-t0:.1f}s")
print(f"Augmented feature rows: {len(aug_features):,}")

### Augmentation report

In [ ]:
augmentation_report(train_df, aug_features)

### Signal comparison: real vs. synthetic F06

Verify that augmented signals are plausible perturbations of the source.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Pick one real F06 training experiment and its first two synthetic counterparts
f06_train_ids = train_df.loc[train_df["Fault_Mode"] == "F06", "Experiment_Index"].tolist()

if f06_train_ids:
    real_id   = f06_train_ids[0]
    real_sig  = long_df[long_df["Experiment_Index"] == real_id].sort_values("Sample_Index")

    # Find augmented experiments derived from the same source
    # (we sample the first 2 augmented F06 experiments for illustration)
    aug_f06_ids = aug_long[aug_long["Fault_Mode"] == "F06"]["Experiment_Index"].unique()[:2]

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(real_sig["Sample_Index"], real_sig["Measurement"],
            color="steelblue", linewidth=2, label=f"Real (exp {real_id})")
    colors = ["tomato", "orange"]
    for aug_id, color in zip(aug_f06_ids, colors):
        aug_sig = aug_long[aug_long["Experiment_Index"] == aug_id].sort_values("Sample_Index")
        ax.plot(aug_sig["Sample_Index"], aug_sig["Measurement"],
                color=color, alpha=0.7, linewidth=1, label=f"Synthetic (exp {aug_id})")
    ax.set_title("F06: real signal vs. augmented copies (1 % Gaussian noise)")
    ax.set_xlabel("Sample Index")
    ax.set_ylabel("Measurement")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No F06 experiments in training set — check split.")

### Full training-set class distribution after augmentation

In [ ]:
FAULT_ORDER = ["N01", "N02", "F01", "F02", "F03", "F04", "F05", "F06"]

train_aug_df = pd.concat([train_df, aug_features], ignore_index=True)

counts_real = train_df["Fault_Mode"].value_counts().reindex(FAULT_ORDER, fill_value=0).rename("Real")
counts_aug  = aug_features["Fault_Mode"].value_counts().reindex(FAULT_ORDER, fill_value=0).rename("Synthetic")
counts_tot  = train_aug_df["Fault_Mode"].value_counts().reindex(FAULT_ORDER, fill_value=0).rename("Total")

summary = pd.concat([counts_real, counts_aug, counts_tot], axis=1)
summary["Imbalance_ratio"] = (summary["Total"].max() / summary["Total"].replace(0, float("nan"))).round(1)
summary

## Save Outputs

In [ ]:
TRAIN_REAL_PATH = PROJECT_ROOT / "data" / "train_real.csv"
TEST_REAL_PATH  = PROJECT_ROOT / "data" / "test_real.csv"
TRAIN_AUG_PATH  = PROJECT_ROOT / "data" / "train_augmented.csv"
AUG_ONLY_PATH   = PROJECT_ROOT / "data" / "augmented_only.csv"

train_df.to_csv(TRAIN_REAL_PATH, index=False)
test_df.to_csv(TEST_REAL_PATH, index=False)
train_aug_df.to_csv(TRAIN_AUG_PATH, index=False)
aug_features.to_csv(AUG_ONLY_PATH, index=False)

print(f"train_real.csv      : {len(train_df):,} rows")
print(f"test_real.csv       : {len(test_df):,} rows")
print(f"train_augmented.csv : {len(train_aug_df):,} rows")
print(f"augmented_only.csv  : {len(aug_features):,} rows")